        # 🎯 L05　過度擬合與偏差－變異
        **統計冒險之旅 2026**　｜　Day 2（09/21 一）⛰️ 模型之嶺　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch2、Ch7 前段；資料：模擬資料


        ### 🎯 這一關你會學到
        - train_test_split：訓練、驗證與測試各有不同任務
- 用驗證集比較多項式次數並畫出 U 型曲線
- 用驗證集比較 KNN 的 K，理解模型彈性

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.2.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L05"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["5-1", "5-2", "5-3", "5-4", "5-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_5_1(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "訓練R2"), 0.90560, 0.005): return (False, "訓練R2 = lm.score(X_train, y_train)；切分要用 test_size=0.25, random_state=42。")
    if not 約等於(抓變數(ns, "測試R2"), 0.82863, 0.005): return (False, "測試R2 = lm.score(X_test, y_test)。")
    return (bool(抓變數(ns, "考古題比較高")) == True, "比較兩個 R²。")
任務定義("5-1", _check_5_1, 提示="score() 用哪一份資料，就是哪一種分數。")

def _l05_expected_rmse(ns):
    """依學員當下套件版本重算基準，避免把數值解或索引寫死。"""
    import numpy as _np
    from sklearn.linear_model import LinearRegression as _LinearRegression
    from sklearn.metrics import mean_squared_error as _mean_squared_error
    from sklearn.pipeline import make_pipeline as _make_pipeline
    from sklearn.preprocessing import PolynomialFeatures as _PolynomialFeatures
    xp_tr = 抓變數(ns, "xp_train")
    xp_va = 抓變數(ns, "xp_valid")
    yp_tr = 抓變數(ns, "yp_train")
    yp_va = 抓變數(ns, "yp_valid")
    train_rmse, valid_rmse = [], []
    for degree in range(1, 16):
        model = _make_pipeline(
            _PolynomialFeatures(degree), _LinearRegression()
        ).fit(xp_tr, yp_tr)
        train_rmse.append(
            _np.sqrt(_mean_squared_error(yp_tr, model.predict(xp_tr)))
        )
        valid_rmse.append(
            _np.sqrt(_mean_squared_error(yp_va, model.predict(xp_va)))
        )
    return _np.asarray(train_rmse), _np.asarray(valid_rmse)

def _check_5_2(run):
    out, ns = run()
    import numpy as _np
    tr = 抓變數(ns, "訓練RMSE們")
    te = 抓變數(ns, "驗證RMSE們")
    if len(tr) != 15 or len(te) != 15:
        return (False, "訓練與驗證都要算 1～15 次，各有 15 個 RMSE。")
    try:
        tr = _np.asarray(tr, dtype=float)
        te = _np.asarray(te, dtype=float)
    except Exception:
        return (False, "RMSE 清單應該都是可比較的數值。")
    if not (_np.isfinite(tr).all() and _np.isfinite(te).all()):
        return (False, "RMSE 清單不能包含 NaN 或無限大。")
    expected_tr, expected_te = _l05_expected_rmse(ns)
    if not _np.allclose(tr, expected_tr, rtol=1e-5, atol=1e-7):
        return (False, "訓練 RMSE 的計算不對：每個次數都要用 yp_train 與模型的訓練預測。")
    if not _np.allclose(te, expected_te, rtol=1e-5, atol=1e-7):
        return (False, "驗證 RMSE 的計算不對：每個次數都要用 yp_valid 與模型的驗證預測。")
    expected_degree = int(_np.argmin(expected_te)) + 1
    return (
        int(抓變數(ns, "最佳次數")) == expected_degree,
        "最佳次數要由目前這份驗證 RMSE 清單的最低點算出（argmin + 1），不必背固定數字。",
    )
任務定義("5-2", _check_5_2, 提示="驗證 RMSE 用 yp_valid 和 p.predict(xp_valid)，最佳次數再由 argmin 取得。")

def _check_5_3(run):
    out, ns = run()
    if not run.figs: return (False, "沒有畫出圖。")
    f = run.figs[0]
    if f["n_lines"] < 2: return (False, "要畫兩條線（訓練與驗證）。")
    if not f["legend"]: return (False, "記得加圖例 plt.legend()。")
    return (("U" in f["title"]) or ("次數" in f["title"]), "標題要包含「U」或「次數」。")
任務定義("5-3", _check_5_3, 提示="兩次 plt.plot 各加 label，再 plt.legend()。")

def _check_5_4(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "驗證RMSE_K1"), 0.68238, 0.01): return (False, "K=1 的驗證 RMSE 不對。")
    if not 約等於(抓變數(ns, "驗證RMSE_K15"), 0.54671, 0.01): return (False, "K=15 的驗證 RMSE 不對。")
    return (int(抓變數(ns, "較好的K")) == 15, "RMSE 小的比較好。")
任務定義("5-4", _check_5_4, 提示="n_neighbors=15。")

def _check_5_5(run):
    out, ns = run()
    import numpy as _np
    gaps = 抓變數(ns, "差距們")
    if len(gaps) != 15:
        return (False, "差距們 要有 15 個。")
    try:
        gaps = _np.asarray(gaps, dtype=float)
    except Exception:
        return (False, "差距們 應該都是可比較的數值。")
    if not _np.isfinite(gaps).all():
        return (False, "差距們 不能包含 NaN 或無限大。")
    expected_tr, expected_te = _l05_expected_rmse(ns)
    expected_gaps = expected_te - expected_tr
    if not _np.allclose(gaps, expected_gaps, rtol=1e-5, atol=1e-7):
        return (False, "每個差距都應該是同一次數的驗證 RMSE 減去訓練 RMSE。")
    expected_degree = int(_np.argmax(expected_gaps)) + 1
    if int(抓變數(ns, "最大差距次數")) != expected_degree:
        return (False, "最大差距次數要由目前這份差距清單算出（argmax + 1），不必背固定數字。")
    return (str(抓變數(ns, "它的問題")) == "高變異", "訓練很好、驗證很差 → 手抖 → 高變異（過度擬合）。")
任務定義("5-5", _check_5_5, 提示="先用 argmax 找目前差距清單的最大值；訓練誤差低、驗證誤差高是高變異訊號。")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
adv = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.0-rc.1/data/advertising.csv")

## 🎯 5-1　考古題 vs 正式考試
L04 的 R² 是用「學過的資料」算的——像拿考古題考自己，當然高。模型真正的實力要看**沒看過的資料**。
做法：把資料切成**訓練集**（考古題，拿來 fit）和**測試集**（正式考試，只拿來評分）。`train_test_split(..., test_size=0.25, random_state=42)`。

In [ ]:
X, y = adv[["TV", "Radio"]], adv["Sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(len(X_train), "筆考古題，", len(X_test), "筆正式考試")
lm = LinearRegression().fit(X_train, y_train)
print("訓練 R²", round(lm.score(X_train, y_train), 3), "| 測試 R²", round(lm.score(X_test, y_test), 3))

## 5-2　過度擬合：把考古題連錯字都背起來
模型越「彈性」，越能貼合訓練資料——連雜訊都一起學進去，換到未見資料反而可能考差。這叫**過度擬合（overfitting）**；相反地太簡單、學不到規則叫**欠擬合**。

實驗：先把一份彎曲的模擬資料切成**訓練集**與**驗證集（validation/dev set）**，再用 1 次到 15 次多項式比較訓練誤差與驗證誤差。

> 🔒 因為我們會反覆查看這份 holdout 來選 degree 與 K，所以它的角色是「驗證集」，不是最終測試集。本關著重概念實驗，不把驗證分數宣稱為最終泛化表現。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
rng = np.random.default_rng(42)
x = np.sort(rng.uniform(0, 6, 100)); y = 2 * np.sin(x) + 0.3 * x + rng.normal(0, 0.5, 100)   # 真實規則是彎的
xp_train, xp_valid, yp_train, yp_valid = train_test_split(x.reshape(-1, 1), y, test_size=0.4, random_state=42)   # 概念實驗：這份 holdout 用來選模型，所以叫驗證集
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
grid = np.linspace(0, 6, 200).reshape(-1, 1)
for a, deg in zip(ax, [1, 5, 15]):
    p = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(xp_train, yp_train)
    a.scatter(xp_train, yp_train, s=12, label="訓練"); a.scatter(xp_valid, yp_valid, s=12, marker="x", label="驗證"); a.plot(grid, p.predict(grid), color="red")
    a.set_ylim(-3, 5); a.set_title(f"{deg} 次多項式：驗證 RMSE {np.sqrt(mean_squared_error(yp_valid, p.predict(xp_valid))):.2f}"); a.legend()
plt.tight_layout(); plt.show()

## 5-3　偏差－變異權衡：準心偏 vs 手抖
- **偏差 bias**：模型太簡單、形狀猜錯（1 次直線去配彎的規則）→ 準心偏掉，怎麼練都打不中。
- **變異 variance**：模型太彈性、對訓練資料太敏感（15 次多項式）→ 換一批資料就整個變樣，像手抖。

期望泛化誤差可概念性拆成偏差²＋變異＋運氣（ε）。把「次數」從 1 掃到 15，驗證誤差常會先降後升，形成 **U 型曲線**；谷底可用來選候選複雜度，但仍不是最終測試成績。

In [ ]:
訓練RMSE們, 驗證RMSE們 = [], []
for deg in range(1, 16):
    p = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(xp_train, yp_train)
    訓練RMSE們.append(np.sqrt(mean_squared_error(yp_train, p.predict(xp_train))))
    驗證RMSE們.append(np.sqrt(mean_squared_error(yp_valid, p.predict(xp_valid))))
plt.plot(range(1, 16), 訓練RMSE們, marker="o", label="訓練"); plt.plot(range(1, 16), 驗證RMSE們, marker="s", label="驗證")
plt.xlabel("多項式次數"); plt.ylabel("RMSE"); plt.title("U 型曲線：驗證誤差先降後升"); plt.legend(); plt.show()
print("驗證誤差最低的次數：", int(np.argmin(驗證RMSE們)) + 1)

## 5-4　KNN：另一種「彈性」的旋鈕
**K 近鄰（KNN）** 不畫線，直接找「最像的 K 個鄰居」取平均。K = 1 只看最近的一個 → 超彈性（手抖）；K 很大 → 平滑（可能準心偏）。K 就是 KNN 的複雜度旋鈕。

本節仍用同一份**驗證集**比較 K，不接觸任何最終測試資料。

In [ ]:
for k in [1, 5, 15, 40]:
    kn = KNeighborsRegressor(n_neighbors=k).fit(xp_train, yp_train)
    print(f"K = {k:>2}：驗證 RMSE {np.sqrt(mean_squared_error(yp_valid, kn.predict(xp_valid))):.3f}")

### 🎯 任務 5-1　切分與兩種分數

用 `TV`、`Radio` 預測 `Sales`：`test_size=0.25, random_state=42` 切分，用線性迴歸算出 `訓練R2` 與 `測試R2`，並把 `考古題比較高` 設成布林值（訓練 R² 是否高於測試 R²）。

In [ ]:
# 🎯 任務 5-1　切分與兩種分數（請保留這一行）
X, y = adv[["TV", "Radio"]], adv["Sales"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=???, random_state=???)
lm = LinearRegression().fit(X_train, y_train)
訓練R2 = ???
測試R2 = ???
考古題比較高 = 訓練R2 > 測試R2
print(round(訓練R2, 3), round(測試R2, 3), 考古題比較高)

In [ ]:
檢查("5-1")   # ◀ 執行這一格，看看任務 5-1 有沒有過關

### 🎯 任務 5-2　多項式次數實驗

沿用 5-2 節的 `xp_train, xp_valid, yp_train, yp_valid`（彎曲資料），對 1～15 次多項式各算訓練與驗證 RMSE，存成串列 `訓練RMSE們`、`驗證RMSE們`；`最佳次數` 是驗證 RMSE 最低的次數。 請用 `np.argmin(驗證RMSE們) + 1` 從目前結果取得答案，不要填入預先記住的固定次數；合理的套件版本可能產生些微數值差異。

In [ ]:
# 🎯 任務 5-2　多項式次數實驗（請保留這一行）
訓練RMSE們, 驗證RMSE們 = [], []
for deg in range(1, 16):
    p = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(xp_train, yp_train)
    訓練RMSE們.append(np.sqrt(mean_squared_error(yp_train, p.predict(xp_train))))
    驗證RMSE們.append(???)
最佳次數 = int(np.argmin(驗證RMSE們)) + 1
print(np.round(驗證RMSE們, 3)); print("最佳次數", 最佳次數)

In [ ]:
檢查("5-2")   # ◀ 執行這一格，看看任務 5-2 有沒有過關

### 🎯 任務 5-3　畫出 U 型曲線

把 `訓練RMSE們` 與 `驗證RMSE們` 對次數（1～15）畫在同一張圖（兩條線、加圖例），標題要包含「U」或「次數」。

In [ ]:
# 🎯 任務 5-3　畫出 U 型曲線（請保留這一行）
plt.plot(range(1, 16), 訓練RMSE們, marker="o", label="訓練")
plt.plot(range(1, 16), ???, marker="s", label="驗證")
plt.xlabel("多項式次數"); plt.ylabel("RMSE"); plt.title(???); plt.legend(); plt.show()

In [ ]:
檢查("5-3")   # ◀ 執行這一格，看看任務 5-3 有沒有過關

### 🎯 任務 5-4　KNN 的 K

用 KNN 迴歸配同一份彎曲資料：算出 `驗證RMSE_K1`（K=1）與 `驗證RMSE_K15`（K=15），並把 `較好的K` 設成 1 或 15。這是在 validation 上選 K，不是最終 test 評估。

In [ ]:
# 🎯 任務 5-4　KNN 的 K（請保留這一行）
kn1 = KNeighborsRegressor(n_neighbors=1).fit(xp_train, yp_train)
kn15 = KNeighborsRegressor(n_neighbors=???).fit(xp_train, yp_train)
驗證RMSE_K1 = np.sqrt(mean_squared_error(yp_valid, kn1.predict(xp_valid)))
驗證RMSE_K15 = ???
較好的K = 1 if 驗證RMSE_K1 < 驗證RMSE_K15 else 15
print(round(驗證RMSE_K1, 3), round(驗證RMSE_K15, 3), 較好的K)

In [ ]:
檢查("5-4")   # ◀ 執行這一格，看看任務 5-4 有沒有過關

### 🎯 任務 5-5　偏差還是變異？

算出每個次數的 `差距們`（驗證 RMSE − 訓練 RMSE，共 15 個），找出差距最大的次數 `最大差距次數`，並把 `它的問題` 設成 `"高變異"`（過度擬合）或 `"高偏差"`（欠擬合）。 `最大差距次數` 請用 `np.argmax(差距們) + 1` 從目前結果取得，不要填入固定數字。

In [ ]:
# 🎯 任務 5-5　偏差還是變異？（請保留這一行）
差距們 = [v - r for v, r in zip(驗證RMSE們, 訓練RMSE們)]
最大差距次數 = int(np.argmax(差距們)) + 1
它的問題 = ???
print(np.round(差距們, 3)); print(最大差距次數, 它的問題)

In [ ]:
檢查("5-5")   # ◀ 執行這一格，看看任務 5-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把雜訊從 0.5 改成 2.0，U 型曲線的谷底會往左還是往右移？為什麼？
2. 把訓練資料從 60 筆改成 600 筆（`rng.uniform(0, 6, 1000)`），15 次多項式還會過度擬合得那麼嚴重嗎？

---
## 🔑 通關密語
　你已經知道為什麼永遠要留一份「正式考試」了。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🚦 L06 分類與評估** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.2.0-rc.1/notebooks/L06_classification.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.2.0-rc.1/